<a href="https://colab.research.google.com/github/AsserGharib1/flyrank-internshipML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsserGharib1/flyrank-internshipML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup, and the same slice as last week

Lane locked: **Lane 2, Refresh and Content Opportunity Scoring**. Nothing in the last three
notebooks made me want to move.

I rebuild the exact slice from ML-04 rather than a new one. March 2026 for everything I am allowed
to know, April 2026 for the outcome, decision date 1 April. The baselines skill is firm that the
rule has to be scored on the same data and the same label the model will later use, otherwise the
comparison in Week 5 means nothing.

The label is only ever used to score the rule. It never goes into it.

In [1]:
%pip -q install duckdb pandas

import os, json, duckdb, pandas as pd, numpy as np

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "No HF_TOKEN. Add it in the Colab secrets panel and turn on notebook access."

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT = BASE + "/fact_content_daily_performance"
FEATURE_MONTH, OUTCOME_MONTH = "2026-03", "2026-04"
DECISION_DATE = pd.Timestamp("2026-04-01")
FLOOR = 100

def month_rel(m):
    return f"read_parquet('{FACT}/month={m}/*.parquet')"

# Walk up to the repo root so the CSV lands in the right place.
previous = None
while not os.path.isdir("work") and os.getcwd() != previous:
    previous = os.getcwd()
    os.chdir("..")
os.makedirs("work/outputs", exist_ok=True)
print("Writing outputs to:", os.path.abspath("work/outputs"))

Writing outputs to: /work/outputs


In [2]:
# Same March into April slice as ML-04, with one change and one thing I could not fix.
#
# My top ten came back with pages sitting at position 0.3, and Search Console positions start
# at 1. I assumed my arithmetic was at fault, since ML-04 built position from gsc_sum_position
# over impressions. So I switched to gsc_avg_position, which the table provides directly, and
# weighted it by impressions. I print both so the difference is visible rather than claimed.

frame = con.sql(f"""
    WITH march AS (
        SELECT client_hash_id AS client_id,
               content_hash_id AS content_id,
               SUM(gsc_impressions) AS impressions_prev30,
               SUM(gsc_clicks)      AS clicks_prev30,
               COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days_prev30,
               SUM(gsc_avg_position * gsc_impressions)
                 FILTER (WHERE gsc_avg_position > 0) * 1.0
                 / NULLIF(SUM(gsc_impressions) FILTER (WHERE gsc_avg_position > 0), 0)
                 AS avg_position_prev30,
               SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)
                 AS avg_position_old_way
        FROM {month_rel(FEATURE_MONTH)}
        GROUP BY 1, 2
    ),
    april AS (
        SELECT client_hash_id AS client_id,
               content_hash_id AS content_id,
               SUM(gsc_impressions) AS impressions_next30
        FROM {month_rel(OUTCOME_MONTH)}
        GROUP BY 1, 2
    )
    SELECT m.*, COALESCE(a.impressions_next30, 0) AS impressions_next30
    FROM march m
    LEFT JOIN april a USING (client_id, content_id)
    WHERE m.impressions_prev30 >= {FLOOR}
    ORDER BY client_id, content_id
""").df()

old_bad = (frame.avg_position_old_way < 1).sum()
new_bad = (frame.avg_position_prev30 < 1).sum()
print("Average position, two ways of building it")
print(f"  ML-04 way, gsc_sum_position over impressions : {old_bad:,} pages below position 1 "
      f"({old_bad/len(frame):.1%}), which is impossible")
print(f"  using gsc_avg_position weighted by impressions: {new_bad:,} pages below position 1")
print(f"  median position, old way {frame.avg_position_old_way.median():.1f}, "
      f"new way {frame.avg_position_prev30.median():.1f}")
print("  the two agree, so the sub-1 positions are in the source column, not in my arithmetic")
print()

# The outcome, exactly as defined in ML-03 and ML-04.
frame["change_pct"] = (frame.impressions_next30 - frame.impressions_prev30) / frame.impressions_prev30 * 100
frame["gap_vs_client"] = frame.change_pct - frame.groupby("client_id").change_pct.transform("median")
frame["fell_behind"] = (frame.gap_vs_client <= -20).astype(int)

BASE_RATE = frame.fell_behind.mean()
print(f"{len(frame):,} pages, {frame.client_id.nunique()} clients")
print(f"base rate, share that fell behind their own site: {BASE_RATE:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Average position, two ways of building it
  ML-04 way, gsc_sum_position over impressions : 707 pages below position 1 (0.7%), which is impossible
  using gsc_avg_position weighted by impressions: 578 pages below position 1
  median position, old way 8.2, new way 8.3
  the two agree, so the sub-1 positions are in the source column, not in my arithmetic

101,441 pages, 44 clients
base rate, share that fell behind their own site: 0.290


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Before writing anything I checked the two signals the rule would lean on. Both sit behind real
FlyRank flags from the session.

### Signal 1, staleness. The one behind the refresh flags.

This was going to be my rule. A page that used to get traffic and has not been touched in months is
the classic refresh candidate, and it is what the refresh flags key off.

So I looked at `content_updated_date` before building anything on it.

In [3]:
# SIGNAL 1 — staleness. Is the update date usable on 1 April at all?

content = con.sql(f"""
    SELECT content_hash_id AS content_id, content_created_date, content_updated_date
    FROM read_parquet('{BASE}/dim_content.parquet')
""").df()

sig1 = frame.merge(content, on="content_id", how="left")
sig1["updated"] = pd.to_datetime(sig1.content_updated_date)

print("The five most common 'last updated' dates in my slice:")
for d, n in sig1.updated.dt.date.value_counts().head(5).items():
    marker = "  <- after my decision date" if pd.Timestamp(d) > DECISION_DATE else ""
    print(f"  {d}   {n:>7,} pages  ({n/len(sig1):5.1%}){marker}")
print()

sig1["days_since_update_at_decision"] = (DECISION_DATE - sig1.updated).dt.days
print(f"Pages whose last update is dated AFTER 1 April: {(sig1.updated > DECISION_DATE).mean():.1%}")
print(f"Median days since update as of 1 April: {sig1.days_since_update_at_decision.median():.0f}")
print("  negative means the edit had not happened yet on the day I am making the decision")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

The five most common 'last updated' dates in my slice:
  2026-05-20    18,766 pages  (18.5%)  <- after my decision date
  2026-02-25    17,324 pages  (17.1%)
  2026-07-03    10,100 pages  (10.0%)  <- after my decision date
  2026-05-18     8,112 pages  ( 8.0%)  <- after my decision date
  2026-06-17     7,000 pages  ( 6.9%)  <- after my decision date

Pages whose last update is dated AFTER 1 April: 82.1%
Median days since update as of 1 April: -61
  negative means the edit had not happened yet on the day I am making the decision


In [4]:
# SIGNAL 1 bucket table, so the verdict rests on numbers and not on a hunch.

sig1["staleness_bucket"] = pd.cut(
    sig1.days_since_update_at_decision,
    bins=[-10_000, -1, 30, 90, 180, 10_000],
    labels=["not updated yet on 1 Apr", "0 to 30 days", "31 to 90 days",
            "91 to 180 days", "over 180 days"])

tbl1 = (sig1.groupby("staleness_bucket", observed=False)
            .agg(n=("fell_behind", "size"), fell_behind_rate=("fell_behind", "mean")).round(3))
print("Staleness at the decision date, against the outcome")
print(tbl1.to_string())
print(f"\nbase rate for comparison: {BASE_RATE:.3f}")

Staleness at the decision date, against the outcome
                              n  fell_behind_rate
staleness_bucket                                 
not updated yet on 1 Apr  83330             0.271
0 to 30 days                 96             0.250
31 to 90 days             17873             0.378
91 to 180 days              124             0.419
over 180 days                18             0.278

base rate for comparison: 0.290


**Signal 1 verdict: FALSE.**

Not false because staleness does not matter. False because this column cannot tell me about
staleness on 1 April.

**82.1 percent of pages carry an update date later than my decision date**, and the median page
shows minus 61 days, meaning the edit happens two months after the moment I am supposed to be
choosing what to review. The common dates cluster in May, June and July, which is when the
warehouse was being built, not when an editor touched the page.

The bucket that swallows most of the slice is literally called "not updated yet on 1 April", and it
holds 83,330 pages. Using this column would mean reading facts from the future and calling them
features. That is the same mistake I made deliberately in ML-04 with April impressions, only this
one is much easier to miss because the column name sounds harmless.

So the refresh style rule is dead, and finding that out here rather than in Week 5 is the whole
value of checking first.

### Signal 2, click through rate against position. The one behind the CTR-fix logic.

Second attempt. A page sitting in a reasonable position that gets fewer clicks than its neighbours
at the same position is the classic CTR-fix candidate.

Position has to be held constant, otherwise I am just measuring position. So each page is compared
only against pages in the same position band.

In [5]:
# SIGNAL 2 — CTR against position. Everything here is March only.

sig2 = frame[frame.avg_position_prev30.notna() & (frame.avg_position_prev30 <= 30)].copy()
sig2["ctr_prev30"] = sig2.clicks_prev30 / sig2.impressions_prev30 * 100
sig2["position_band"] = pd.cut(sig2.avg_position_prev30, bins=[0, 3, 5, 10, 20, 30],
                               labels=["1 to 3", "3 to 5", "5 to 10", "10 to 20", "20 to 30"])

curve = (sig2.groupby("position_band", observed=False)
             .agg(n=("ctr_prev30", "size"), clicks=("clicks_prev30", "sum"),
                  impressions=("impressions_prev30", "sum")))
curve["ctr_pct"] = (curve.clicks / curve.impressions * 100).round(2)
print("First, does click through rate actually fall with position in this data?")
print(curve[["n", "ctr_pct"]].to_string())
print()

band_median = sig2.groupby("position_band", observed=False).ctr_prev30.transform("median")
sig2["ctr_gap"] = sig2.ctr_prev30 - band_median
sig2["ctr_gap_bucket"] = pd.qcut(sig2.ctr_gap, 5,
    labels=["worst fifth", "second", "middle", "fourth", "best fifth"], duplicates="drop")

tbl2 = (sig2.groupby("ctr_gap_bucket", observed=False)
            .agg(n=("fell_behind", "size"), fell_behind_rate=("fell_behind", "mean")).round(3))
print("Then, how far below its position peers a page sits, against the outcome")
print(tbl2.to_string())
print(f"\nbase rate for comparison: {BASE_RATE:.3f}")

First, does click through rate actually fall with position in this data?
                   n  ctr_pct
position_band                
1 to 3          9755     0.39
3 to 5         17084     0.35
5 to 10        30726     0.30
10 to 20       19721     0.32
20 to 30       10810     0.19

Then, how far below its position peers a page sits, against the outcome
                    n  fell_behind_rate
ctr_gap_bucket                         
worst fifth     17621             0.409
second          21258             0.328
middle          13982             0.274
fourth          17616             0.222
best fifth      17619             0.183

base rate for comparison: 0.290


**Signal 2 verdict: CONFIRMED, with one caveat I want on the record.**

The second table is what I was hoping for. Sort pages by how far their click through rate sits below
their position peers, and the share that later falls behind drops cleanly from about **0.41 in the
worst fifth to about 0.18 in the best**, with no wobble in between, on roughly 17,000 pages per
bucket against a base rate of 0.29. The worst fifth is more than twice as likely to fall behind as
the best fifth.

The caveat is the first table. In real search the click through rate at position 1 to 3 is worth
tens of percent and it collapses as you go down the page. Here it runs from about 0.4 at the top to
about 0.2 at position 20 to 30. Almost flat, and tiny throughout. I saw the same flatness on the
starter file back in ML-02, so it is not a one off.

I cannot explain that from this data, and I am not going to pretend I can. What it means practically
is that the position adjustment in my rule is doing very little work, because the bands barely
differ. The signal is really "low click through rate for a page with impressions", and the position
banding is more insurance than correction. That is worth saying out loud rather than claiming a
position adjustment that is not really adjusting anything.

### The rule, in plain words

> A page is worth reviewing first if it gets a lot of impressions and hardly anyone clicks it,
> compared with other pages sitting at the same position.

Coded, that is how far below its band it sits, multiplied by how visible it is. Nothing fitted,
nothing from April, nothing built from the label.

* **Reason code:** `below_ctr_for_position_band`
* **Action label:** `review_title_and_meta`

One reason code, as the card asks. The action follows from it. If a page has the audience and is not
getting the click, the thing a human can actually change that day is the title and the description.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# The rule. Deliberately readable.

q = sig2.copy()
q["ctr_shortfall"] = (-q.ctr_gap).clip(lower=0)      # only pages BELOW their peers score
q["visibility"]    = np.log1p(q.impressions_prev30)  # the same gap matters more on a busy page
q["score"]         = (q.ctr_shortfall * q.visibility).round(3)

q["reason_code"]  = "below_ctr_for_position_band"
q["action_label"] = "review_title_and_meta"

queue = q.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

COLS = ["rank", "score", "reason_code", "action_label", "client_id", "content_id",
        "impressions_prev30", "clicks_prev30", "ctr_prev30", "avg_position_prev30",
        "position_band", "ctr_gap", "active_days_prev30"]

CSV_PATH = "work/outputs/baseline_action_score.csv"
queue[COLS].to_csv(CSV_PATH, index=False)
print(f"Wrote {len(queue):,} ranked rows to {CSV_PATH}")
print("That CSV stays out of git by design. The notebook rebuilds it on every run.")

Wrote 88,096 ranked rows to work/outputs/baseline_action_score.csv
That CSV stays out of git by design. The notebook rebuilds it on every run.


In [7]:
# Score the rule. This is the only place the label is touched.

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

print(f"base rate: {BASE_RATE:.3f}   (what picking at random gets you)")
print()
pk = {}
for k in (10, 20, 50, 100):
    p = precision_at_k(queue.score, queue.fell_behind, k)
    pk[f"p_at_{k}"] = round(p, 4)
    print(f"  precision at {k:>3}: {p:.3f}   ({p*k:.0f} of the top {k} really did fall behind)"
          f"   lift {p/BASE_RATE:.1f}x")

metrics = {
    "slice": {"feature_month": FEATURE_MONTH, "outcome_month": OUTCOME_MONTH,
              "decision_date": str(DECISION_DATE.date()), "min_march_impressions": FLOOR,
              "rows": int(len(queue)), "clients": int(queue.client_id.nunique())},
    "label": {"definition": "April change at least 20 points below the median page on the same site",
              "base_rate": round(float(BASE_RATE), 4)},
    "rule": {"plain_words": "high impressions, low clicks for its position band",
             "score": "ctr_shortfall_vs_position_band * log1p(march_impressions)",
             "reason_code": "below_ctr_for_position_band",
             "action_label": "review_title_and_meta"},
    "signal_verdicts": {"staleness": "FALSE", "ctr_vs_position": "CONFIRMED"},
    "precision_at_k": pk,
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print()
print("Receipts written to work/outputs/baseline_metrics.json. The CSV stays out of git,")
print("this JSON goes in. Printing it here too so the numbers live inside the notebook.")
print()
print(json.dumps(metrics, indent=2))
print()
print("Precision at 10 of", metrics["precision_at_k"]["p_at_10"], "is what Week 5 has to beat.")

base rate: 0.290   (what picking at random gets you)

  precision at  10: 0.900   (9 of the top 10 really did fall behind)   lift 3.1x
  precision at  20: 0.700   (14 of the top 20 really did fall behind)   lift 2.4x
  precision at  50: 0.560   (28 of the top 50 really did fall behind)   lift 1.9x
  precision at 100: 0.560   (56 of the top 100 really did fall behind)   lift 1.9x

Receipts written to work/outputs/baseline_metrics.json. The CSV stays out of git,
this JSON goes in. Printing it here too so the numbers live inside the notebook.

{
  "slice": {
    "feature_month": "2026-03",
    "outcome_month": "2026-04",
    "decision_date": "2026-04-01",
    "min_march_impressions": 100,
    "rows": 88096,
    "clients": 43
  },
  "label": {
    "definition": "April change at least 20 points below the median page on the same site",
    "base_rate": 0.29
  },
  "rule": {
    "plain_words": "high impressions, low clicks for its position band",
    "score": "ctr_shortfall_vs_position_band *

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The card asks for ten written up. I print twenty so the pattern is visible.

In [8]:
# Read my own top of the list, with the answer column showing.

review = queue.head(20)[["rank", "score", "impressions_prev30", "clicks_prev30", "ctr_prev30",
                         "avg_position_prev30", "position_band", "active_days_prev30",
                         "fell_behind"]].copy()
review["ctr_prev30"] = review.ctr_prev30.round(3)
review["avg_position_prev30"] = review.avg_position_prev30.round(1)
print("Top 20 by score. fell_behind is the answer, never an input.")
print(review.to_string(index=False))
print()
print(f"of these 20, {review.fell_behind.sum()} really did fall behind "
      f"(random picking would give about {BASE_RATE*20:.0f})")
print(f"clients represented in the top 20: {queue.head(20).client_id.nunique()}")
print()

# Two things I want counted rather than eyeballed.
sub1_top20 = (review.avg_position_prev30 < 1).sum()
sub1_all   = (queue.avg_position_prev30 < 1).mean()
print(f"rows in the top 20 with an impossible position below 1: {sub1_top20}"
      f"   ({sub1_top20/20:.0%} of the top 20, against {sub1_all:.1%} of the whole queue)")
print("so the odd rows are heavily over represented at the top, see section 4")

Top 20 by score. fell_behind is the answer, never an input.
 rank  score  impressions_prev30  clicks_prev30  ctr_prev30  avg_position_prev30 position_band  active_days_prev30  fell_behind
    1  2.579            134984.0            1.0       0.001                  2.7        1 to 3                  31            0
    2  2.560            124075.0            1.0       0.001                  0.3        1 to 3                  31            1
    3  2.549            212404.0           24.0       0.011                  0.7        1 to 3                  31            1
    4  2.470             83834.0            1.0       0.001                  0.1        1 to 3                  31            1
    5  2.452             38865.0            0.0       0.000                  4.8        3 to 5                  31            1
    6  2.411             48049.0            4.0       0.008                  4.2        3 to 5                  31            1
    7  2.397            143019.0           4

### The top ten, one line each

Every one of these carries the same reason code and the same action, because the rule only emits
one. So what is worth writing is the confidence and what would break it.

1. **Review title and meta.** A very large audience and essentially nobody clicking. Wrong if those
   impressions come from queries the page was never meant to answer, in which case no title fixes it.
2. **Review title and meta.** Same shape as row 1. Wrong if a sibling page on the same site is
   taking the clicks, which my data cannot see because I never joined at topic level.
3. **Review title and meta.** The biggest audience anywhere in the queue. Wrong if that scale means
   the page is a hub that people reach some other way.
4. **Review title and meta.** Wrong if the demand behind it is seasonal and March happened to be
   its peak, so April was always going to look worse.
5. **Review title and meta.** Zero clicks in the whole month against a real audience. Wrong if the
   page answers something readable straight off the results page, where no click is expected.
6. **Review title and meta.** Wrong if this rate is simply normal for its topic, which would mean
   my band median is the wrong yardstick for it.
7. **Review title and meta.** Highest click count in the top ten, so the gap here is relative rather
   than absolute. Wrong if the absolute clicks are healthy and only the ratio looks poor.
8. **Review title and meta.** Fewer active days than the rest of the group, so March is less
   complete for this page. Wrong if it only went live partway through the month.
9. **Review title and meta.** Wrong if the audience is branded, where the click tends to go to the
   homepage instead of the page that showed.
10. **Review title and meta.** Smallest audience in the top ten and therefore the least certain of
    the group. Wrong if the click through rate is just noisy at this volume.

The honest summary is that these ten share one failure mode. Every one assumes the impressions
represent real intent that a better title could capture. If the impressions are junk, the whole top
of the queue is junk, and nothing in this data can tell me which it is.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Three weak picks, and the first one made me change the code.

**The impossible positions, and the wrong explanation I nearly published.** My top ten came back
with pages at position 0.3, 0.7 and 0.1. Search Console positions start at 1, so those cannot be
real.

I assumed the fault was mine, since ML-04 built position out of `gsc_sum_position` divided by
impressions. So I rewrote it to use `gsc_avg_position`, which the table gives directly, weighted by
impressions.

It made almost no difference. The two versions land on a median of **8.2 and 8.3**, and the direct
column still puts **578 pages below position 1** against 707 for my version. The sub-1 values are in
the warehouse column itself. My arithmetic was not the problem, and the explanation I was about to
write down would have been wrong.

I have kept the direct column, because using the value the table provides is easier to defend than
rebuilding it. But I cannot explain positions below 1, and I am not going to invent a reason.

What matters for the queue is the concentration. Sub-1 positions are about **0.6 percent** of the
slice, yet three of them turned up in my top twenty. My score multiplies the gap by visibility, and
these rows carry enormous impression counts, so whatever is odd about them is exactly the thing my
rule reaches for. Those three rows are not trustworthy and I would pull them before handing this
list to anyone.

This only surfaced because the card made me read individual rows. No summary statistic would have
shown it.

**Rank 1 did not fall behind.** The single highest scoring page in my queue is a miss. It has a very
large audience and one click, so the rule loves it, but its April traffic held up fine. That is a
useful reminder that a big click through gap is not the same thing as a page in trouble.

**The top of the queue is concentrated.** A handful of clients own most of the top twenty. That is
the same concentration problem I flagged in ML-04, where one client was about a third of the pool. A
reviewer handed this list would spend their week on two or three sites. A real queue needs a per
client cap, which is a Week 5 fix.

**Leakage check** below. The score touches three columns, all March. Nothing from April, nothing
built from the label, and the staleness column stays out because signal 1 showed it is dated after
the decision.

In [9]:
# Leakage check on every column the rule actually touches.

BUILT_FROM = ["clicks_prev30", "impressions_prev30", "avg_position_prev30"]
NEVER_USED = {
    "impressions_next30": "the outcome window itself",
    "change_pct":         "derived from the outcome",
    "gap_vs_client":      "derived from the outcome",
    "fell_behind":        "this is the label",
    "content_updated_date": "dated after the decision, see signal 1",
    "search_volume":      "snapshot value, as of date never verified",
}

print("What the score is built from:")
for c in BUILT_FROM:
    print(f"  {c:<22} March 2026 only, complete before 1 April")
print()
print("What is deliberately kept out:")
for c, why in NEVER_USED.items():
    print(f"  {c:<22} {why}")
print()
print("Hard check that none of the excluded columns reached the score:")
leaked = [c for c in NEVER_USED if c in BUILT_FROM]
print(f"  columns that leaked in: {leaked if leaked else 'none'}")
print()
print("And the CSV a reviewer would receive carries no outcome column:")
outcome_cols = [c for c in COLS if c in NEVER_USED]
print(f"  outcome columns present in the CSV: {outcome_cols if outcome_cols else 'none'}")

What the score is built from:
  clicks_prev30          March 2026 only, complete before 1 April
  impressions_prev30     March 2026 only, complete before 1 April
  avg_position_prev30    March 2026 only, complete before 1 April

What is deliberately kept out:
  impressions_next30     the outcome window itself
  change_pct             derived from the outcome
  gap_vs_client          derived from the outcome
  fell_behind            this is the label
  content_updated_date   dated after the decision, see signal 1
  search_volume          snapshot value, as of date never verified

Hard check that none of the excluded columns reached the score:
  columns that leaked in: none

And the CSV a reviewer would receive carries no outcome column:
  outcome columns present in the CSV: none


## Self-check

Before you submit, confirm each line honestly:

* [x] Every section above is filled, thinking written out and code that backs it
* [x] The notebook runs top to bottom with no errors (Runtime, then Run all)
* [x] No client names, web addresses, or private search terms anywhere, and no token in any cell
* [x] My claims stay careful. Observed, measured, provisional, decision support
* [x] Committed under `work/notebooks/`, then submit the repo URL on the card